# Can we predict `arrears_next_month`?
Question: using what is known at the end of a billing month, can we predict whether the account is in arrears the following month?

Plan: inspect data, check for leakage, validate over time (train on earlier months, test on the latest), compare with baselines.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score, RocCurveDisplay
from sklearn.inspection import permutation_importance
df = pd.read_csv('data.csv', parse_dates=['billing_month'])
print(df.shape, df.meter_ref.nunique(), 'accounts;', df.billing_month.min().date(), '->', df.billing_month.max().date())
print(df.isna().sum().sum(), 'missing;', df.duplicated(['meter_ref','billing_month']).sum(), 'duplicate account-months')
df.describe().T

In [ ]:
print('Base rate of arrears:', df.arrears_next_month.mean().round(3))
for c in ['tariff','occupants','late_payments_12m','final_notice_sent']:
    print(df.groupby(c).arrears_next_month.agg(['mean','size']).round(3), '\n')
print(df.groupby('billing_month').arrears_next_month.mean().round(3).to_string())

## Leakage check: `final_notice_sent`
Per the dictionary it is set *in the billing cycle that followed* the billing month, i.e. **after** the prediction point, and is plausibly a consequence of the account falling into arrears (arrears rate ~87% with a notice vs ~1% without). It would not be available at prediction time, so it is **excluded**. It is used below only to show how much it would inflate results.

## Model
Features: tariff, occupants, kwh, bill_eur, late_payments_12m, plus implied price per kWh. Split by time: the last 3 months are the test set. A random row split would be too optimistic since accounts repeat.

In [ ]:
df['eur_per_kwh'] = df.bill_eur / df.kwh
cat, num = ['tariff'], ['occupants','kwh','bill_eur','late_payments_12m','eur_per_kwh']
cut = df.billing_month.sort_values().unique()[-3]
tr, te = df[df.billing_month < cut], df[df.billing_month >= cut]
print(len(tr), len(te), 'share of test accounts also in train:', te.meter_ref.isin(tr.meter_ref).mean().round(2))
y_tr, y_te = tr.arrears_next_month, te.arrears_next_month
def pre(): return ColumnTransformer([('c', OneHotEncoder(), cat), ('n', StandardScaler(), num)])
models = {'logistic': make_pipeline(pre(), LogisticRegression(max_iter=1000)),
          'gbm': make_pipeline(pre(), HistGradientBoostingClassifier(random_state=0))}
res = {}
for n, m in models.items():
    m.fit(tr[cat+num], y_tr); p = m.predict_proba(te[cat+num])[:,1]; res[n]=p
    print(f'{n:9s} ROC-AUC {roc_auc_score(y_te,p):.3f}  PR-AUC {average_precision_score(y_te,p):.3f}')
print('base rate (PR-AUC of a no-skill model):', y_te.mean().round(3))
print('late_payments_12m alone: ROC-AUC', round(roc_auc_score(y_te, te.late_payments_12m),3))

In [ ]:
ax = plt.gca()
for n,p in res.items(): RocCurveDisplay.from_predictions(y_te, p, name=n, ax=ax)
ax.plot([0,1],[0,1],'k:'); ax.set_title('Test ROC (without final_notice_sent)'); plt.show()
pi = permutation_importance(models['gbm'], te[cat+num], y_te, scoring='roc_auc', n_repeats=10, random_state=0)
pd.Series(pi.importances_mean, index=cat+num).sort_values().plot.barh(title='Permutation importance (drop in test AUC)'); plt.show()

In [ ]:
p = res['gbm']
for t in [0.2,0.3,0.4,0.5]:
    pred = p>=t; tp=(pred&(y_te==1)).sum()
    print(f'thr {t}: precision {tp/max(pred.sum(),1):.2f}  recall {tp/y_te.sum():.2f}  flagged {pred.mean():.0%}')
q = pd.qcut(p, 10, duplicates='drop')
print(pd.DataFrame({'pred':p,'obs':y_te}).groupby(q, observed=True).mean().round(3))

In [ ]:
# Illustration only: adding the leaky feature
cols = cat+num+['final_notice_sent']
m2 = make_pipeline(ColumnTransformer([('c',OneHotEncoder(),cat),('n','passthrough',num+['final_notice_sent'])]), HistGradientBoostingClassifier(random_state=0))
m2.fit(tr[cols], y_tr); print('with final_notice_sent: ROC-AUC', round(roc_auc_score(y_te, m2.predict_proba(te[cols])[:,1]),3), '(not a usable result)')

## Conclusion
- **Partly.** Without `final_notice_sent`, the models rank accounts reasonably (test ROC-AUC ~0.74, PR-AUC ~0.55 vs 0.25 no-skill). It is a useful prioritisation signal, not a precise forecast.
- **Main driver is `late_payments_12m`** (arrears rate 7% with 0 late payments, 34% with 2, ~56% with 3+); occupants has a small effect. Tariff, kWh and bill amount add little.
- **`final_notice_sent` is leakage** (recorded after the prediction point, AUC 0.965 with it). Do not use it.
- **The arrears rate is not stable over time**: ~3% through 2024, rising to ~26% by Dec 2025. Models trained on earlier months under-predict on the test period (see the calibration table), so probabilities need recalibration on recent data, and performance should be monitored. Ranking quality holds up, absolute levels do not.
- Caveats: one time-split test (3 months), no hyperparameter tuning, and 93% of test accounts also appear in training. Next steps: rolling-origin validation, per-account history features (e.g. trend in late payments), and a check on why arrears jumped in 2025.